In [1]:
from cgra import *
from kernels import *
import random

In [2]:
kernel_name = "benchmarks/disco-vs-oe/softmax_partial"
version = ""

In [3]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [4]:
# Data
def configMemory(data, nRows, nCols):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # ------------------------------------------
    # CONFIGURATION
    # ------------------------------------------
    # LoopnIt                f_0        -   f_1
    # &maxValues + 4         &in + 12   -   maxValInit
    # &d                     -          -  -
    # -------------------------------------------

    first_addr_in = first_addr
    first_addr_maxVals = first_addr_in + nRows * nCols * 4
    addr_d = first_addr_maxVals + nCols * 4

    loopnIt = (nCols - 1) // 4
    taylor_reciprocal_factors = [ int(1 * pow(2,24)), int(1/2 * pow(2,24))] 
    maxInit = -200

    config_vals_col0 = [ loopnIt, first_addr_maxVals + 4, addr_d]
    config_vals_col1 = [ taylor_reciprocal_factors[0], first_addr_in + 12]
    config_vals_col2 = [ ]
    config_vals_col3 = [ taylor_reciprocal_factors[1], maxInit]

    addr = 0
    for cfg in [config_vals_col0, config_vals_col1,
                config_vals_col2, config_vals_col3]:
        kernel_add_memory_region(kernel_name, addr, cfg, version=version)
        addr += len(cfg) * 4

    kernel_add_memory_region(kernel_name, first_addr_in, data, version=version)

    return [
        0,
        len(config_vals_col0) * 4,
        (len(config_vals_col0) + len(config_vals_col1)) * 4,
        (len(config_vals_col0) + len(config_vals_col1) + len(config_vals_col2)) * 4
    ]

In [5]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT", "R1", "INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [6]:
def getResult(first_addr_C, end_addr_C, size):
    result = [0 for _ in range(size)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [7]:
NUM_FRACTION_BITS = 12
def MUL(x, y):
        return (x * y) >> NUM_FRACTION_BITS
    
N_TAYLOR_COEFF = 3
reciprocal_factorials = [
    int(1 * pow(2,24)),   # 1/1!
    int(1/2 * pow(2,24))    # 1/2!
]

def exp_fixed_point_taylor(input_val):

    result = 1 << NUM_FRACTION_BITS
    x_pow_n = input_val

    for i in range(N_TAYLOR_COEFF - 1):
        result += MUL(x_pow_n, reciprocal_factorials[i])
        x_pow_n = MUL(x_pow_n, input_val)

    return result

def softmax_partial_cpu(data, i, nCols):
    maxValues = [0 for _ in range(nCols)]
    maxValues[0] = data[0]
       
    d = 0
    for j in range(1, nCols):

        idx = i * nCols + j

        current_val = data[idx]

        max_val = current_val if current_val > maxValues[j-1] else maxValues[j-1]
        maxValues[j] = max_val

        exp_term = exp_fixed_point_taylor(current_val - max_val)

        shift = max_val - maxValues[j-1]

        if shift >= 0:
            d = (d >> shift) + exp_term
        else:
            d = (d << (-shift)) + exp_term

        data[idx] = exp_term

    return data, maxValues


In [8]:
# Test dimensions (cols must be multiple of 4 + 1) 
nRows = 1
nCols = 121

data = [random.randint(-10, 10) for _ in range(nRows * nCols)]
data_cpy = data.copy()
data_cpy_cpu = data.copy()

load_addrs = configMemory(data_cpy, nRows, nCols)

In [ ]:
# Row by row
for i in range(nRows):
    expected_data_res, expected_maxVals_res = softmax_partial_cpu(data_cpy_cpu, i, nCols)
    runKernel(load_addrs, max_it=20000)

    # Get result from CGRA
    addr_out_data = first_addr + i * nCols * 4
    cgra_out_data = getResult(addr_out_data, addr_out_data + nCols * 4, nCols)
    
    addr_out_maxVals = first_addr + nRows * nCols * 4
    cgra_out_maxVals = getResult(addr_out_maxVals, addr_out_maxVals + nCols * 4, nCols)

    # Check result correctness
    print("Checking data...")
    errors = 0
    for i in range(len(expected_data_res)):
        if expected_data_res[i] != cgra_out_data[i]:
            errors += 1
    if errors > 0:
        print(f"Rows {i}: Errors in data: {errors} out of {nCols}")
        print(f"Expected: {expected_data_res}")
        print(f"CGRA: {cgra_out_data}")
    else:
        print("OK")

    print("Checking maxVals...")
    errors = 0
    for i in range(len(expected_maxVals_res)):
        if expected_maxVals_res[i] != cgra_out_maxVals[i]:
            errors += 1
    if errors > 0:
        print(f"Rows {i}: Errors in maxVals: {errors} out of {nCols}")
        print(f"Expected: {expected_maxVals_res}")
        print(f"CGRA: {cgra_out_maxVals}")
    else:
        print("OK")